# Inspect Consolidated SWE Datasets

Load and visualize the four SWE source datasets at a single peak-winter day, in their **native consolidated form** (the per-source gridded NCs / zarrs that the aggregator reads). This is the per-source counterpart to the cross-source HRU view in [inspect_target_swe.ipynb](../targets/inspect_target_swe.ipynb).

Sources (see `catalog/variables.yml` → `snow_water_equivalent` and `catalog/sources.yml`):

- **Daymet V4 R1 `swe`** — daily, 1 km LCC over North America. Native units `kg m-2` (≡ mm). Distributed as a multi-year zarr store; this notebook reads the operator-staged path from caldera. CRS is Lambert Conformal Conic — not lat/lon.
- **SNODAS `swe`** — daily, ~1 km WGS84 over CONUS. Native units `kg m-2` (≡ mm). Distributed as daily `.tar`; consolidated to per-year NetCDFs at `<datastore>/snodas/daily/`.
- **ERA5-Land `sd`** — daily, 0.1° (~11 km) global. Native units `m` of water equivalent (instantaneous, `cell_methods: "time: point"`). Daily NCs at `<datastore>/era5_land/daily/`.
- **Margulis WUS-SR `SWE`** — daily, ~480 m over the Western US only (`fabric_scope.fabrics: [or]`). Native units `m` of water equivalent. Consolidated to per-calendar-year NCs at `<datastore>/margulis_wus_sr/daily/`.

The first plot shows raw values in their native units and native grids — Daymet in LCC, the others in lat/lon. The 'Normalized comparison' section below converts every source to **inches** (PRMS `pkwater_equiv` units; `÷ 25.4` for mm sources, `× 1000 / 25.4` for m sources) and clips to a common CONUS bbox so the magnitudes are visually comparable on a shared colour scale.

**Daymet path note.** The 4.5 TB Daymet zarr lives at an operator-staged location, not under `<datastore>`. The default path in this notebook matches the gfv2-spatial-targets project's `manifest.json`; edit `DAYMET_NA_ZARR` if your project registers a different path.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

from _helpers import save_figure

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT_DIR = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")
DAYMET_NA_ZARR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/data_creation/daymet/source/zarr/daymet_na.zarr"
)

# Set True (and re-run) to populate docs/figures/consolidated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = True
_helpers.PROJECT = PROJECT_DIR.name

TARGET_DATE = "2010-03-01"  # near-peak CONUS SWE
TARGET_YEAR = 2010

## Load all four datasets at TARGET_DATE

Per-source notes:

- **Daymet** uses `xr.open_zarr` and selects one day; this is fast because the zarr is chunked along time.
- **SNODAS / ERA5-Land sd / Margulis** open the per-year NC and `.sel(time=TARGET_DATE, method='nearest')` (Daymet's noon-timestamping convention also means a `nearest` selection picks the right day).
- Each source records its **native units** in the loaded dict alongside the conversion factor that brings it to inches. The conversion is applied later in the normalized-comparison cell.

In [ ]:
datasets = {
    "Daymet V4 R1 (swe, LCC)": {
        "path": DAYMET_NA_ZARR,
        "open": "zarr",
        "var": "swe",
        "units": "kg m-2 (mm)",
        "to_inches": lambda da: da / 25.4,
    },
    "SNODAS (swe, WGS84)": {
        "path": DATASTORE / "snodas" / "daily" / f"snodas_daily_{TARGET_YEAR}.nc",
        "open": "nc",
        "var": "swe",
        "units": "kg m-2 (mm)",
        "to_inches": lambda da: da / 25.4,
    },
    "ERA5-Land (sd, WGS84)": {
        "path": DATASTORE / "era5_land" / "daily" / f"era5_land_daily_{TARGET_YEAR}.nc",
        "open": "nc",
        "var": "sd",
        "units": "m (water-eq)",
        "to_inches": lambda da: da * 1000.0 / 25.4,
    },
    "Margulis WUS-SR (SWE, WUS-only)": {
        "path": DATASTORE / "margulis_wus_sr" / "daily" / f"margulis_wus_sr_daily_{TARGET_YEAR}.nc",
        "open": "nc",
        "var": "SWE",
        "units": "m (water-eq)",
        "to_inches": lambda da: da * 1000.0 / 25.4,
    },
}

opened = {}
for label, info in datasets.items():
    path = info["path"]
    if not path.exists():
        print(f"SKIP {label}: {path} not found (run fetch/consolidate first)")
        continue
    if info["open"] == "zarr":
        ds = xr.open_zarr(path, chunks={})
    else:
        ds = xr.open_dataset(path)
    opened[label] = (ds, info)
    print(
        f"{label}: {list(ds.data_vars)[:6]}{'...' if len(ds.data_vars) > 6 else ''}"
        f" | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}"
    )

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)

## Plot at TARGET_DATE — native units, native grids

Four panels, each in its source's native CRS:

- Daymet shows in LCC (the x/y axes are projected metres) — the CONUS coastline will look slightly curved compared to the lat/lon panels.
- SNODAS / ERA5-Land / Margulis show in WGS84 lat/lon.

**What to look for.** All four should show the same broad pattern: deepest snowpack in the western mountains (Sierra Nevada / Cascades / Rockies), substantial snow across the Northern Plains and Northeast, and essentially no snow south of ~35° N. Margulis covers **only the Western US**; everything east of the Rockies will appear blank by design (this is documented in the catalog's `fabric_scope`). ERA5-Land's coarse 11 km grid will smooth out fine-scale ridge/valley contrast that SNODAS and Daymet capture at ~1 km.

**Red flags.** A unimodal map with no snow in the Rockies in early March is a smoking gun for missed `_FillValue` decoding or a wrong reducer; a Margulis panel showing values outside California / Sierra / Nevada is a fetch bug.

In [ ]:
def _select_day(ds, var):
    """Select TARGET_DATE on the source's native time axis. `nearest` is
    robust to Daymet's noon-of-day timestamping convention vs. SNODAS /
    ERA5-Land / Margulis at start-of-day or end-of-day.
    """
    return ds[var].sel(time=TARGET_DATE, method="nearest").load()


if not opened:
    print("No datasets available yet. Run the fetch / consolidate commands first.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    flat = axes.flatten()
    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = flat[idx]
        da = _select_day(ds, info["var"])
        actual_time = str(da.time.values)[:10]
        da.plot(ax=ax, cmap="Blues", robust=True)
        ax.set_title(
            f"{label}\n{actual_time} | {info['units']}",
            fontsize=11,
        )
        ax.set_aspect("equal")
    for idx in range(len(opened), 4):
        flat[idx].set_visible(False)
    fig.suptitle(
        f"SWE consolidated sources — raw native units / native grids — {TARGET_DATE}",
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    save_figure(fig, "swe_raw_panels")
    plt.show()

## Normalized comparison — inches, shared SNODAS footprint

Convert every source to **inches** (PRMS `pkwater_equiv` units) and place
Daymet, SNODAS, and ERA5-Land onto a **common spatial footprint**: the
SNODAS grid (EPSG:5070, ~1 km, CONUS extent set by PR #155's pre-projection
step). Margulis is left in its native WGS84 lat/lon at ~480 m so its panel
shows the source's true WUS footprint without resampling.

Unit chain per source:

- Daymet (`kg m-2` ≡ mm) → ÷ 25.4 → inches → reproject LCC → EPSG:5070 (SNODAS grid)
- SNODAS (`kg m-2` ≡ mm) → ÷ 25.4 → inches (already on the reference grid)
- ERA5-Land (`m`) → × 1000 → mm → ÷ 25.4 → inches → reproject WGS84 → EPSG:5070
- Margulis (`m`) → × 1000 → mm → ÷ 25.4 → inches → keep native WGS84

`rioxarray.reproject_match` handles CRS conversion, resampling, and clipping
to the SNODAS grid in one call. Daymet's CRS is decoded from its
`lambert_conformal_conic` grid mapping variable via `pyproj.CRS.from_cf`
(mirrors `aggregate/daymet.py:_crs_from_grid_mapping`). ERA5-Land's 11 km
pixels are upsampled to the 1 km SNODAS grid — coarse texture is preserved,
only the apparent pixel size shrinks. Colour scale is anchored on the
2nd / 98th percentile of SNODAS to keep the deep-snowpack tail from washing
out the rest of the colour ramp.

In [ ]:
import numpy as np
import rioxarray  # noqa: F401  enables the .rio accessor
from pyproj import CRS as PyprojCRS

# Margulis kept in native lat/lon; clip to a WUS-only bbox for display.
MARGULIS_LON_RANGE = (-125.0, -102.0)
MARGULIS_LAT_RANGE = (31.0, 49.0)


def _wrap_lon_to_minus180_180(da, lon_dim):
    lon = da[lon_dim]
    if float(lon.max()) > 180:
        new_lon = ((lon + 180) % 360) - 180
        da = da.assign_coords({lon_dim: new_lon}).sortby(lon_dim)
    return da


def _clip_margulis(da):
    """Crop Margulis to its WUS bbox (kept in native WGS84 lat/lon)."""
    lon_dim = "lon" if "lon" in da.dims else "longitude"
    lat_dim = "lat" if "lat" in da.dims else "latitude"
    da = _wrap_lon_to_minus180_180(da, lon_dim)
    lat_vals = da[lat_dim].values
    if lat_vals[0] > lat_vals[-1]:
        lat_slice = slice(MARGULIS_LAT_RANGE[1], MARGULIS_LAT_RANGE[0])
    else:
        lat_slice = slice(MARGULIS_LAT_RANGE[0], MARGULIS_LAT_RANGE[1])
    return da.sel({lon_dim: slice(*MARGULIS_LON_RANGE), lat_dim: lat_slice})


def _source_crs(label, ds):
    if "Daymet" in label:
        # Decode LCC from the grid mapping variable (mirrors aggregate/daymet.py).
        return PyprojCRS.from_cf(ds["lambert_conformal_conic"].attrs).to_wkt()
    if "SNODAS" in label:
        return "EPSG:5070"  # set by PR #155 pre-projection in fetch/snodas.py
    if "ERA5-Land" in label:
        return "EPSG:4326"
    if "Margulis" in label:
        return "EPSG:4326"
    raise ValueError(f"Unknown source label: {label}")


if opened:
    snodas_label = next((lbl for lbl in opened if "SNODAS" in lbl), None)
    margulis_label = next((lbl for lbl in opened if "Margulis" in lbl), None)

    if snodas_label is None:
        raise RuntimeError("SNODAS not loaded — needed as the reference footprint.")

    # Build the reference grid first (SNODAS day, in inches, on EPSG:5070).
    snodas_ds, snodas_info = opened[snodas_label]
    snodas_day = _select_day(snodas_ds, snodas_info["var"])
    snodas_day = snodas_day.rio.write_crs(_source_crs(snodas_label, snodas_ds))
    snodas_in = snodas_info["to_inches"](snodas_day).rio.write_crs("EPSG:5070")

    normalized = {}
    for label, (ds, info) in opened.items():
        da_in = info["to_inches"](_select_day(ds, info["var"]))
        if label == snodas_label:
            normalized[label] = (snodas_in, info)
        elif label == margulis_label:
            normalized[label] = (_clip_margulis(da_in), info)
        else:
            # Daymet (LCC) or ERA5-Land (WGS84) → reproject onto SNODAS grid.
            da_in = da_in.rio.write_crs(_source_crs(label, ds))
            normalized[label] = (da_in.rio.reproject_match(snodas_in), info)

    # Colour scale anchored on SNODAS percentiles (the reference footprint).
    ref_vals = snodas_in.values.ravel()
    ref_vals = ref_vals[~np.isnan(ref_vals)]
    vmin = float(np.percentile(ref_vals, 2))
    vmax = float(np.percentile(ref_vals, 98))
    print(f"Colour scale (from SNODAS, 2nd/98th pct): {vmin:.2f} – {vmax:.2f} inches")

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    flat = axes.flatten()
    for idx, (label, (da, info)) in enumerate(normalized.items()):
        ax = flat[idx]
        actual_time = str(da.time.values)[:10]
        da.plot(ax=ax, cmap="Blues", vmin=vmin, vmax=vmax)
        if label == margulis_label:
            footprint = "native WGS84 (WUS bbox)"
        else:
            footprint = "EPSG:5070 SNODAS grid"
        ax.set_title(
            f"{label}\n{actual_time} | inches (PRMS units) | {footprint}",
            fontsize=11,
        )
        ax.set_aspect("equal")
    for idx in range(len(normalized), 4):
        flat[idx].set_visible(False)
    fig.suptitle(
        f"SWE consolidated sources — inches, shared SNODAS footprint — {TARGET_DATE}\n"
        f"Daymet/ERA5 reprojected onto SNODAS grid; Margulis kept in native WGS84",
        fontsize=13, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, "swe_normalized_comparison")
    plt.show()

    print()
    print(f"{'Source':<40} {'mean (in)':>12} {'95th pct (in)':>14}")
    print("-" * 70)
    for label, (da, _) in normalized.items():
        finite = da.values[~np.isnan(da.values)]
        if finite.size == 0:
            print(f"{label:<40} {'(all NaN)':>12} {'-':>14}")
        else:
            print(
                f"{label:<40} {float(finite.mean()):>12.3f} "
                f"{float(np.percentile(finite, 95)):>14.3f}"
            )

### Why do the four SWE sources differ?

Even after rescaling to inches and cropping to a common bbox, the four panels will show visibly different magnitudes and spatial textures. None of these is a bug — each reflects a real difference in how the source was produced.

1. **Different production lineages.**
   - **Daymet** is a daily-gridded climatology, produced by spatial interpolation of NWS COOP / SNOTEL precipitation gauges + a temperature-based snow-fraction model. SWE is a derived diagnostic, not a direct observation.
   - **SNODAS** is an operational analysis: a snow model forced by AWIPS hourly precipitation, assimilating airborne / satellite / ground SWE observations. Closest thing to a "truth" product over CONUS but only available 2003-present.
   - **ERA5-Land** is a global reanalysis (ECMWF IFS land model forced by ERA5 atmosphere). No SWE assimilation; pure model.
   - **Margulis WUS-SR** is a 32-year (1985–2017) snow reanalysis specifically for the WUS, assimilating Landsat snow-covered area into an ensemble snow model. Highest spatial fidelity of the four sources within its footprint; coarsest temporal coverage (no operational stream).

2. **Different spatial resolution.**
   - Daymet ~1 km LCC, SNODAS ~1 km WGS84, Margulis ~480 m, ERA5-Land ~11 km. ERA5-Land's 100× coarser pixel area systematically under-resolves the deepest mountain ridges — its 95th-percentile SWE will look noticeably lower than the others at the same percentile.

3. **Different spatial extent.**
   - SNODAS and Daymet are CONUS-wide.
   - ERA5-Land is global (we crop to CONUS for the comparison panel).
   - Margulis WUS-SR is **Western US only** — its panel will be blank east of the Rockies by design. The catalog records this via `fabric_scope.fabrics: [or]` and the SWE target builder honours it (Margulis only contributes on the OR fabric).

4. **Different temporal coverage.**
   - Daymet: 1980–2024. SNODAS: 2003-09-30 to present. ERA5-Land: 1950-present. Margulis WUS-SR: 1985-2017 (water years).
   - The SWE target builder honours these via NaN-aware min/max — outside any single source's coverage, the bound reduces to whatever sources have valid data.

5. **Different timestamp conventions.**
   - Daymet at **noon UTC** of each day; SNODAS at **06 UTC**; ERA5-Land at **00 UTC** of a stop-of-window snapshot; Margulis at **start-of-day**. A `nearest` selection against a single date handles this; an exact-match `.sel(time=...)` would silently fail on at least one source.

**Calibration target implication.** The SWE target builder reads each source's HRU-aggregated NC (the `aggregated/` notebooks visualize that next step), converts each to inches, and takes the **NaN-aware min / max across sources at each (HRU, day)**. The lower / upper bound the calibrator targets is the spread across these four lineages — an HRU with deep maritime snowpack might see SNODAS at 80 in and ERA5-Land at 30 in on the same day, and the calibrator is asked to land *between* those, not at either extreme. NN-fill happens at target stage on the combined bound, never on the aggregated NC.

The raw / normalized panels in this notebook are for spatial-pattern sanity-checking each input; the [inspect_target_swe.ipynb](../targets/inspect_target_swe.ipynb) notebook drives the per-HRU post-combination view.

## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()